# Módulo 1 · 05 — Preguntas de negocio: flota y conductores

Para la ruta $r$ y el día $t$ (parámetros en `config/modulo1.yaml`):

$$D_{r,t} = \left\lceil \frac{\hat y_{r,t}}{C \cdot \rho} \right\rceil, \qquad
V_r = \left\lfloor \frac{H_{op}}{T^{ciclo}_r} \right\rfloor, \qquad
N_{r,t} = \left\lceil \frac{\max(D^{ida}_{r,t}, D^{reg}_{r,t})}{V_r} \right\rceil$$

$$K_{r,t} = \left\lceil N_{r,t}\cdot \left\lceil \frac{H_{op}}{J_{max}} \right\rceil \cdot \frac{7}{d_{sem}} \cdot \frac{1}{1-a} \right\rceil$$

$C$: capacidad, $\rho$: ocupación objetivo, $H_{op}$: horas de operación, $T^{ciclo}$: ida+regreso, $J_{max}$: jornada máxima (norma citada), $d_{sem}$: días laborables por semana, $a$: ausentismo.

In [ ]:
# parameters
DEMO = False
OCUPACION = None   # sobrescribe config si no es None
USAR_CUANTIL = True

In [ ]:
import sys
from pathlib import Path
RAIZ = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(RAIZ))
sys.path.insert(0, str(RAIZ / "modulo1_demanda"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from common.seeds import fijar_semilla, semillas_experimento
from common.config import cargar_config
import m1lib as m1

fijar_semilla()
cfg = cargar_config("modulo1")
M1 = RAIZ / "modulo1_demanda"
DATA, FIG, MOD = M1 / "data", M1 / "figures", M1 / "models"
H, TAU = cfg["pronostico"]["horizonte"], cfg["pronostico"]["cuantil_alto"]
plt.rcParams["figure.figsize"] = (12, 4)

Y = pd.read_parquet(DATA / "processed" / "series_diarias.parquet")       # pasajeros/ruta/día (imputada)
cal = pd.read_parquet(DATA / "processed" / "calendario.parquet")
rutas = pd.read_csv(DATA / "processed" / "rutas.csv", dtype=str)
INI_VAL, INI_TEST = pd.Timestamp(cfg["datos"]["inicio_validacion"]), pd.Timestamp(cfg["datos"]["inicio_prueba"])
RUTAS_IDA = rutas.loc[rutas.sentido == "ida", "ruta"].tolist()
if DEMO:
    RUTAS_IDA = RUTAS_IDA[:cfg["demo"]["rutas"]]
Y_ida = Y[RUTAS_IDA]
origenes_test = m1.origenes_rodantes(Y.index, INI_TEST, H, cfg["pronostico"]["paso_origen"])
escalas_mase = {r: m1.escala_mase(Y_ida.loc[:INI_VAL - pd.Timedelta(days=1), r].dropna().values) for r in RUTAS_IDA}
print(f"{len(RUTAS_IDA)} rutas · {len(origenes_test)} orígenes de prueba: {origenes_test[0].date()} … {origenes_test[-1].date()}")

op = cfg["operacion"]
if OCUPACION is not None:
    op["ocupacion_objetivo"] = OCUPACION
CAP = op["capacidad_por_clase"][op["clase_dimensionamiento"]]
RHO, HOP, AUS = op["ocupacion_objetivo"], op["horas_operacion"], op["ausentismo"]
JMAX = op["jornada_conductor"]["horas_max_dia"]
ciclo = lambda r: op["tiempo_ciclo_horas"].get(r, op["tiempo_ciclo_defecto"])
regreso_de = dict(zip(rutas.ruta_par[rutas.sentido == "regreso"], rutas.ruta[rutas.sentido == "regreso"]))
todos = pd.read_parquet(DATA / "processed" / "pred_todos.parquet")

## 1. Flota y conductores para los próximos 30 días

Se usa el último origen disponible. El sentido de regreso, si no tiene pronóstico propio, se aproxima con su demanda observada de la misma semana del año anterior escalada (_TODO: o entrenar también las rutas de regreso_).

In [ ]:
MODELO = "secuencial"
o = todos.origen.max()
plan = []
for r in RUTAS_IDA:
    g = todos[(todos.modelo == MODELO) & (todos.ruta == r) & (todos.origen == o)].set_index("fecha")
    dem = g.q_alto if USAR_CUANTIL else g.pred
    d_ida = m1.despachos_requeridos(dem, CAP, RHO)
    rr = regreso_de.get(r)
    if rr is not None:
        ratio = (Y[rr] / Y[r]).loc[:o].tail(365).median()
        d_reg = m1.despachos_requeridos(dem * ratio, CAP, RHO)
    else:
        d_reg = d_ida
    V = m1.vueltas_por_vehiculo(HOP, ciclo(r))
    veh = m1.vehiculos_requeridos(d_ida, d_reg, V)
    cond = m1.conductores_requeridos(veh, HOP, JMAX, AUS, op["dias_trabajo_semana"])
    plan.append(pd.DataFrame({"ruta": r, "fecha": g.index, "demanda_plan": dem.values, "despachos_ida": d_ida,
                              "despachos_regreso": d_reg, "vehiculos": veh, "conductores": cond}))
plan = pd.concat(plan)
plan.to_csv(DATA / "processed" / "plan_flota.csv", index=False)
plan.groupby("ruta")[["despachos_ida", "vehiculos", "conductores"]].agg(["mean", "max"])

In [ ]:
tot = plan.groupby("fecha")[["vehiculos", "conductores"]].sum()
tot.plot(kind="bar", figsize=(13, 4), title="Flota y conductores requeridos por día (30 días)")
plt.xticks(range(len(tot)), [d.strftime("%d-%b") for d in tot.index], rotation=90)
plt.tight_layout(); plt.savefig(FIG / "fig11_flota_conductores.png", dpi=150)

## 2. Sensibilidad: demanda ±10 % y parámetros operativos

In [ ]:
def flota_total(f_dem=1.0, rho=RHO, cap=CAP, hop=HOP, cic=None):
    tot = 0
    for r in RUTAS_IDA:
        dem = plan.loc[plan.ruta == r, "demanda_plan"].values * f_dem
        d = m1.despachos_requeridos(dem, cap, rho)
        tot += m1.vehiculos_requeridos(d, d, m1.vueltas_por_vehiculo(hop, cic or ciclo(r))).max()
    return tot

base = flota_total()
var = np.linspace(-.2, .2, 9)
sens = pd.DataFrame({
    "demanda": [flota_total(f_dem=1 + v) for v in var],
    "ocupación": [flota_total(rho=RHO * (1 + v)) for v in var],
    "capacidad": [flota_total(cap=CAP * (1 + v)) for v in var],
    "horas_operación": [flota_total(hop=HOP * (1 + v)) for v in var],
    "tiempo_ciclo": [flota_total(cic=op["tiempo_ciclo_defecto"] * (1 + v)) for v in var],
}, index=(var * 100).round()) / base - 1
sens.plot(marker="o", title="Sensibilidad de la flota pico (cambio relativo)"); plt.xlabel("variación del parámetro (%)")
plt.savefig(FIG / "fig12_sensibilidad.png", dpi=150); (sens * 100).round(1)

## 3. Valor del pronóstico: puntual vs P90, red vs referencia

In [ ]:
filas = []
for m in todos.modelo.unique():
    g = todos[todos.modelo == m]
    for base_dem in ["pred", "q_alto"]:
        sin_at, sobr = 0.0, 0.0
        for r, gr in g.groupby("ruta"):
            a, b = m1.costo_operativo(gr.y.values, gr[base_dem].values, CAP, RHO, m1.vueltas_por_vehiculo(HOP, ciclo(r)))
            sin_at += a; sobr += b
        filas.append({"modelo": m, "dimensiona_con": base_dem, "pasajeros_sin_atender": sin_at, "vehiculos_dia_sobrantes": sobr})
valor = pd.DataFrame(filas)
valor

In [ ]:
# Costo total según el costo relativo (pasajero no atendido vs vehículo-día sobrante)
ratios = np.logspace(-2, 1, 30)   # costo pasajero / costo vehículo-día
curvas = {f"{r.modelo}·{r.dimensiona_con}": r.pasajeros_sin_atender * ratios + r.vehiculos_dia_sobrantes for r in valor.itertuples()}
pd.DataFrame(curvas, index=ratios).plot(logx=True, logy=True, title="Costo total vs costo relativo de quedarse corto")
plt.xlabel("costo pasajero no atendido / costo vehículo-día sobrante")

## 4. Temporada alta vs semana normal

_TODO: escoger la temporada alta del horizonte de prueba (p. ej. 15 dic–10 ene) y una semana normal; comparar vehículos y conductores; recomendar flota propia/arrendada/convenios._

In [ ]:
ALTA = ("2025-12-15", "2026-01-10")
NORMAL = ("2025-09-08", "2025-09-14")
g = todos[todos.modelo == MODELO]
def requerimiento(ini, fin):
    s = g[(g.fecha >= ini) & (g.fecha <= fin)].groupby(["ruta", "fecha"]).q_alto.mean().reset_index()
    s["veh"] = [m1.vehiculos_requeridos(*(2 * [m1.despachos_requeridos([d], CAP, RHO)]), m1.vueltas_por_vehiculo(HOP, ciclo(r)))[0]
                for r, d in zip(s.ruta, s.q_alto)]
    v = s.groupby("fecha").veh.sum().max()
    return v, m1.conductores_requeridos([v], HOP, JMAX, AUS, op["dias_trabajo_semana"])[0]
va, ca = requerimiento(*ALTA); vn, cn = requerimiento(*NORMAL)
pd.DataFrame({"vehículos pico": [vn, va, va - vn], "conductores": [cn, ca, ca - cn]}, index=["normal", "temporada alta", "adicionales"])